# Training a Simple Neural Network: Salary Predictor

This notebook trains **SalaryNet**, a tiny feed-forward neural network that predicts salary (in thousands of rupees) from years of experience.

**Architecture:** `1 input -> 2 hidden neurons (ReLU) -> 1 output`

At the end we **save the trained weights** to `salary_net.pth` so the Flask app can serve predictions without retraining.

## 1. Imports

In [1]:
import torch
from torch import nn

from model import SalaryNet, MODEL_PATH

/Users/laxminarayen/Documents/SimpleNeuralNetwork/venv/lib/python3.13/site-packages/torch/_subclasses/functional_tensor.py:279: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:81.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


## 2. Our training data

- `X` = years of experience (0 to 10)
- `y` = salary in **thousands of rupees**

In [2]:
X = torch.arange(0, 11, dtype=torch.float32).reshape(-1, 1)

y = torch.tensor([
    20, 25, 30, 35, 50, 65,
    80, 95, 110, 125, 140
], dtype=torch.float32).reshape(-1, 1)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: torch.Size([11, 1])
y shape: torch.Size([11, 1])


## 3. Build the neural network

The architecture is defined once in `model.py` and imported here, so the training notebook and the Flask app use the **exact same network**.

In [3]:
model = SalaryNet()
print(model)

SalaryNet(
  (hidden): Linear(in_features=1, out_features=2, bias=True)
  (relu): ReLU()
  (output): Linear(in_features=2, out_features=1, bias=True)
)


## 4. Set useful starting values

These are **not** the final trained parameters, just a sensible starting point that helps this tiny demo converge nicely in class.

In [4]:
with torch.no_grad():
    model.hidden.weight.copy_(torch.tensor([[1.2], [0.8]]))
    model.hidden.bias.copy_(torch.tensor([0.4, -1.6]))
    model.output.weight.copy_(torch.tensor([[3.0, 7.0]]))
    model.output.bias.fill_(15.0)

## 5. Define the loss and optimizer

- **Loss:** Mean Squared Error (how far predictions are from the truth)
- **Optimizer:** Adam (adjusts the weights to reduce the loss)

In [5]:
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.02)

## 6. Train the neural network

Each epoch: predict -> measure loss -> compute gradients -> nudge the weights.

In [6]:
for epoch in range(3000):
    prediction = model(X)
    loss = loss_fn(prediction, y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 500 == 0:
        print(f"Epoch {epoch + 1}, Loss: {loss.item():.6f}")

Epoch 500, Loss: 0.530058
Epoch 1000, Loss: 0.042046


Epoch 1500, Loss: 0.001076
Epoch 2000, Loss: 0.000007


Epoch 2500, Loss: 0.000000
Epoch 3000, Loss: 0.000000


## 7. Make a prediction

Predict the salary for someone with 5 years of experience. We multiply by 1000 because the model works in thousands of rupees.

In [7]:
model.eval()

with torch.no_grad():
    experience = torch.tensor([[5.0]])
    predicted_salary = model(experience).item() * 1000

print(f"Predicted salary for 5 years: Rs {predicted_salary:,.0f}")

Predicted salary for 5 years: Rs 65,000


## 8. Save the trained model

We save only the learned weights (`state_dict`). The Flask app rebuilds the same `SalaryNet` and loads these weights.

In [8]:
torch.save(model.state_dict(), MODEL_PATH)
print(f"Model saved to {MODEL_PATH}")

Model saved to /Users/laxminarayen/Documents/SimpleNeuralNetwork/salary_net.pth
